In [83]:
import pandas as pd
import joblib


df = pd.read_csv("../../data/processed/cleaned_housing.csv")

In [84]:
import numpy as np

df['price'] = np.log1p(df['price'])

In [85]:
cols = ['mainroad','guestroom','basement','hotwaterheating','airconditioning','prefarea']
print(df[cols].dtypes)

mainroad           int64
guestroom          int64
basement           int64
hotwaterheating    int64
airconditioning    int64
prefarea           int64
dtype: object


In [86]:
df = pd.get_dummies(df, columns=['furnishingstatus'], drop_first=True)

In [87]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 533 entries, 0 to 532
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   price                            533 non-null    float64
 1   area                             533 non-null    float64
 2   bedrooms                         533 non-null    int64  
 3   bathrooms                        533 non-null    int64  
 4   stories                          533 non-null    int64  
 5   mainroad                         533 non-null    int64  
 6   guestroom                        533 non-null    int64  
 7   basement                         533 non-null    int64  
 8   hotwaterheating                  533 non-null    int64  
 9   airconditioning                  533 non-null    int64  
 10  parking                          533 non-null    int64  
 11  prefarea                         533 non-null    int64  
 12  furnishingstatus_semi-

In [88]:
# Create better features (helps model learn patterns)
df['area_per_room'] = df['area'] / df['bedrooms'].replace(0, np.nan)
df['area_per_room'] = df['area_per_room'].fillna(df['area_per_room'].median())
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['area_squared'] = df['area'] ** 2

df['log_area'] = np.log1p(df['area'])
df['bed_bath_interaction'] = df['bedrooms'] * df['bathrooms']
df['is_new_or_renov_like'] = (df['stories'] >= 2).astype(int)  # proxy feature


In [89]:
X = df.drop('price', axis=1)
y = df['price']

In [90]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.001),
    "RF": RandomForestRegressor(random_state=42),
    "GBR": GradientBoostingRegressor(random_state=42)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = -cross_val_score(
        model, X, y,
        scoring="neg_root_mean_squared_error",
        cv=cv
    )
    print(f"{name}: RMSE mean={scores.mean():.4f}, std={scores.std():.4f}")

Linear: RMSE mean=0.2009, std=0.0245
Ridge: RMSE mean=0.2020, std=0.0243
Lasso: RMSE mean=0.2036, std=0.0231


c:\Users\hp\miniconda3\envs\fayza-ai\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.44454e-18): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\hp\miniconda3\envs\fayza-ai\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.50376e-18): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\hp\miniconda3\envs\fayza-ai\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.57666e-18): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\hp\miniconda3\envs\fayza-ai\lib\site-packages\sklearn\linear_model\_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.3214e-18): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
c:\Users\hp\miniconda3\envs\f

RF: RMSE mean=0.2171, std=0.0139
GBR: RMSE mean=0.2082, std=0.0228


In [91]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0],
    "min_samples_leaf": [1, 3, 5]
}

gbr = GradientBoostingRegressor(random_state=42)

grid = GridSearchCV(
    estimator=gbr,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

preds = best_model.predict(X_test)

Best Params: {'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 3, 'n_estimators': 200, 'subsample': 0.8}
Best CV RMSE: 0.19964395669273677


In [92]:
preds = best_model.predict(X_test)

In [93]:

# Compare
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": preds
})

results.head()


,Actual,Predicted
0,16.046600,15.804787
1,14.819813,15.065164
2,15.622159,15.543867
3,14.845130,15.134872
4,14.960689,15.158565


In [94]:
pd.DataFrame({"y_test": y_test, "preds": preds}).to_csv("../../data/processed/preds.csv", index=False)

In [95]:
joblib.dump(best_model, "../../models/best_model.pkl")
X_test.to_csv("../../data/processed/X_test.csv", index=False)
y_test.to_csv("../../data/processed/y_test.csv", index=False)